In [3]:
import datetime as dt
import platform
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Mapping

import numpy as np
import pandas as pd
import xarray as xr
import torch

import earthkit.data as ekd
import earthkit.regrid as ekr

from anemoi.inference.outputs.printer import print_state
from anemoi.inference.runners.simple import SimpleRunner
from ecmwf.opendata import Client as OpendataClient
from IPython.display import display

In [19]:
@dataclass(frozen=True)
class NotebookConfig:
    source: str = "ecmwf"

    # None usa la corrida más reciente.
    # Ejemplo: "2026-06-10T00:00:00Z"
    date: str | None = "2010-06-10T06:00:00Z"

    # lead_time_hours: es el intervalo 
    # de pronosticos deseados
    lead_time_hours: int = 6
    device: str = "cuda"

    run_download: bool = True
    run_inference: bool = True

    # En esta parte es necesario sustituir 
    # con variables necesarias a solicitar
    output_variables: tuple[str, ...] = (
        "msl", "10u", "10v", "t_850"
    )

    # México, Golfo de México y mares adyacentes.
    # Longitudes en 0–360.
    lat_min: float = 5.0
    lat_max: float = 35.0
    lon_min: float = 240.0
    lon_max: float = 300.0

    output_dir: Path = Path("./outputs/aifs_single_v2")


CONFIG = NotebookConfig()
display(pd.Series(asdict(CONFIG), name="value").to_frame())

,value
source,ecmwf
date,2010-06-10T06:00:00Z
lead_time_hours,6
device,cuda
run_download,True
run_inference,True
output_variables,"(msl, 10u, 10v, t_850)"
lat_min,5.0
lat_max,35.0
lon_min,240.0


In [21]:
def parse_utc_datetime(value: str) -> dt.datetime:
    """
    Convierte una fecha en texto a un objeto ``datetime`` en UTC sin zona horaria.

    Acepta cadenas en formato ISO, incluyendo fechas terminadas en ``Z``.
    Si la fecha incluye zona horaria, se convierte a UTC. Finalmente, la fecha
    se ajusta al inicio de la hora, eliminando minutos, segundos y microsegundos.

    Parámetros
    ----------
    value : str
        Fecha en formato ISO.

    Retorna
    -------
    dt.datetime
        Fecha normalizada en UTC, sin información explícita de zona horaria.
    """

    parsed = dt.datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is not None:
        parsed = parsed.astimezone(dt.timezone.utc).replace(tzinfo=None)
    return parsed.replace(minute=0, second=0, microsecond=0)


def resolve_initial_date(config: NotebookConfig) -> dt.datetime:
    """
    Define y valida la fecha inicial del pronóstico.

    Si la configuración no contiene una fecha explícita, se usa la fecha más
    reciente disponible desde la fuente de datos. Si la fecha fue definida por
    el usuario, se convierte a formato UTC. La función valida que la hora
    corresponda a un ciclo sinóptico permitido: 00, 06, 12 o 18 UTC.

    Parámetros
    ----------
    config : NotebookConfig
        Configuración del notebook, incluyendo la fecha solicitada y la fuente
        de datos.

    Retorna
    -------
    dt.datetime
        Fecha inicial válida para ejecutar AIFS.

    Raises
    ------
    ValueError
        Se lanza si la hora no corresponde a un ciclo sinóptico válido.
    """

    if config.date is None:
        resolved = OpendataClient(config.source).latest()
    else:
        resolved = parse_utc_datetime(config.date)

    if resolved.hour not in {0, 6, 12, 18}:
        raise ValueError(
            f"{resolved.hour:02d} UTC no es una hora sinóptica válida."
        )

    return resolved


DATE = resolve_initial_date(CONFIG)
print(f'Valor de la fecha desde la configuracion inicial: {CONFIG.date}')
print(f'Valor de la fecha en formato ISO: {DATE}')

PREVIOUS_DATE = DATE - dt.timedelta(hours=6)

print("t-6 h:", PREVIOUS_DATE)
print("t0   :", DATE)

Valor de la fecha desde la configuracion inicial: 2010-06-10T06:00:00Z
Valor de la fecha en formato ISO: 2010-06-10 06:00:00
t-6 h: 2010-06-10 00:00:00
t0   : 2010-06-10 06:00:00


In [7]:
ekd.config.set({"cache-policy": "user"})


# ============================================================
# 1. Diccionarios Open Data shortName -> ERA5/CDS variable name
# ============================================================

PARAM_SFC_CDS = {
    "10u": "10m_u_component_of_wind",
    "10v": "10m_v_component_of_wind",
    "2d": "2m_dewpoint_temperature",
    "2t": "2m_temperature",
    "msl": "mean_sea_level_pressure",
    "skt": "skin_temperature",
    "sp": "surface_pressure",
    "tcw": "total_column_water",
    "lsm": "land_sea_mask",
    "z": "geopotential",
    "slor": "slope_of_sub_gridscale_orography",
    "sdor": "standard_deviation_of_orography",
}

PARAM_SOIL_CDS = {
    "vsw": {
        1: "volumetric_soil_water_layer_1",
        2: "volumetric_soil_water_layer_2",
    },
    "sot": {
        1: "soil_temperature_level_1",
        2: "soil_temperature_level_2",
    },
}

PARAM_PL_CDS = {
    "gh": "geopotential",
    "t": "temperature",
    "u": "u_component_of_wind",
    "v": "v_component_of_wind",
    "w": "vertical_velocity",
    "q": "specific_humidity",
}


CDS_DATASETS = {
    "sfc": "reanalysis-era5-single-levels",
    "soil": "reanalysis-era5-single-levels",
    "pl": "reanalysis-era5-pressure-levels",
}

In [31]:
def _as_list(x: str | Iterable[str]) -> list[str]:
    """
    Convierte un string o iterable de strings en lista.
    """
    if isinstance(x, str):
        return [x]
    return list(x)

def _build_cds_request(
    *,
    valid_date: dt.datetime,
    variables: list[str],
    levelist: list[int] | None = None,
    grid: list[float] | tuple[float, float] = (0.25, 0.25),
    area: list[float] | None = None,
) -> dict:
    """
    Construye un request para CDS.

    Para ERA5 pressure levels se añade la clave `pressure_level`.
    Para ERA5 single levels no se usa `pressure_level`.
    """

    request = {
        "product_type": "reanalysis",
        "variable": variables,
        "year": f"{valid_date.year:04d}",
        "month": f"{valid_date.month:02d}",
        "day": f"{valid_date.day:02d}",
        "time": f"{valid_date.hour:02d}:00",
        "grid": list(grid),
        "data_format": "grib",
        "download_format": "unarchived",
    }

    if levelist:
        request["pressure_level"] = [str(level) for level in levelist]

    if area is not None:
        request["area"] = area  # [North, West, South, East]

    return request

def _normalise_field_name(
    *,
    field,
    source: str,
) -> str:
    """
    Normaliza los nombres devueltos por ERA5/CDS para que sean compatibles
    con la convención usada por el pipeline de AIFS.

    Ejemplos esperados:
    - surface:
        2t, 10u, msl, z, slor, sdor
    - pressure levels:
        gh_850, t_850, u_850, v_850, w_850, q_850
    - soil:
        vsw_1, vsw_2, sot_1, sot_2
    """

    short_name = field.metadata(
        "shortName",
        default=field.metadata("param", default="unknown"),
    )

    if source == "sfc":
        return short_name

    if source == "pl":
        level = field.metadata("levelist")
        return f"{short_name}_{level}"
    
    if source == "soil":
        return short_name
    
    raise ValueError(f"source no reconocido: {source!r}")

def _translate_params_to_cds(
    *,
    source: str,
    param: str | Iterable[str],
    levelist: Iterable[int] | None = None,
) -> tuple[str, list[str], list[int]]:
    """
    Traduce parámetros estilo Open Data/MARS a nombres de variables CDS.

    source:
        "sfc"  -> ERA5 single levels
        "soil" -> ERA5 single levels, variables de suelo por capa
        "pl"   -> ERA5 pressure levels
    """

    params = _as_list(param)
    requested_levels = list(levelist) if levelist is not None else []

    if source not in CDS_DATASETS:
        raise ValueError(
            f"source debe ser uno de {list(CDS_DATASETS)}, pero se recibió: {source!r}"
        )

    dataset = CDS_DATASETS[source]

    if source == "sfc":
        missing = [p for p in params if p not in PARAM_SFC_CDS]
        if missing:
            raise ValueError(f"Parámetros surface no definidos para CDS: {missing}")

        variables = [PARAM_SFC_CDS[p] for p in params]

        if requested_levels:
            raise ValueError(
                "source='sfc' no debe recibir levelist. "
                "Los campos surface de ERA5/CDS son single-level."
            )

        return dataset, variables, []

    if source == "pl":
        missing = [p for p in params if p not in PARAM_PL_CDS]
        if missing:
            raise ValueError(f"Parámetros pressure-level no definidos para CDS: {missing}")

        if not requested_levels:
            raise ValueError(
                "source='pl' requiere levelist, por ejemplo "
                "[1000, 925, 850, 700, ..., 50]."
            )

        variables = [PARAM_PL_CDS[p] for p in params]

        return dataset, variables, requested_levels

    if source == "soil":
        missing = [p for p in params if p not in PARAM_SOIL_CDS]
        if missing:
            raise ValueError(f"Parámetros soil no definidos para CDS: {missing}")

        if not requested_levels:
            raise ValueError(
                "source='soil' requiere levelist, por ejemplo [1, 2]. "
                "En CDS las capas de suelo son variables distintas."
            )

        variables = []
        for p in params:
            for level in requested_levels:
                try:
                    variables.append(PARAM_SOIL_CDS[p][level])
                except KeyError as exc:
                    raise ValueError(
                        f"No existe mapeo CDS para {p=} con soil level {level}."
                    ) from exc

        return dataset, variables, []

In [17]:
def get_data(
    *,
    date: dt.datetime,
    source: str,
    param: str | Iterable[str],
    levelist: Iterable[int] | None = None,
    grid: list[float] | tuple[float, float] = (0.25, 0.25),
    area: list[float] | None = None,
    **kwargs,
) -> dict[str, np.ndarray]:
    """
    Recupera campos meteorológicos ERA5/CDS para los tiempos t-6 h y t0.

    Esta versión está adaptada a CDS mediante earthkit:

        ekd.from_source("cds", dataset, request)

    Parámetros
    ----------
    date : dt.datetime
        Fecha de inicialización del pronóstico, correspondiente al tiempo t0.

    source : {"sfc", "soil", "pl"}
        Grupo de variables a recuperar:
        - "sfc": ERA5 single levels.
        - "soil": ERA5 soil variables en single levels.
        - "pl": ERA5 pressure levels.

    param : str | Iterable[str]
        Parámetro o lista de parámetros en formato Open Data/MARS:
        por ejemplo "10u", "2t", "gh", "q", "vsw".

    levelist : Iterable[int] | None, optional
        Para source="pl", niveles de presión.
        Para source="soil", niveles de suelo [1, 2].
        Para source="sfc", debe ser None.

    grid : list[float] | tuple[float, float]
        Resolución solicitada a CDS. Para AIFS se usa normalmente (0.25, 0.25)
        antes de interpolar a N320.

    area : list[float] | None
        Área CDS en formato [North, West, South, East].
        Para AIFS global debe dejarse en None.

    **kwargs
        Argumentos adicionales opcionales. Se incorporan al request CDS.

    Retorna
    -------
    dict[str, np.ndarray]
        Diccionario con arreglos de forma aproximada:

            {
                variable_name: np.ndarray con dimensión temporal [t-6, t0]
            }

        Cada campo es interpolado de ERA5 0.25° regular a N320.
    """

    dataset, cds_variables, requested_levels = _translate_params_to_cds(
        source=source,
        param=param,
        levelist=levelist,
    )

    collected: defaultdict[str, list[np.ndarray]] = defaultdict(list)

    for valid_date in (date - dt.timedelta(hours=6), date):
        print(f'Descargando datos para valid date: {valid_date}')
        request = _build_cds_request(
            valid_date=valid_date,
            variables=cds_variables,
            levelist=requested_levels if source == "pl" else None,
            grid=grid,
            area=area,
        )

        request.update(kwargs)

        data = ekd.from_source(
            "cds",
            dataset,
            request,
        )

        for field in data:
            values = field.to_numpy()

            if values.shape != (721, 1440):
                raise ValueError(
                    "Se esperaba una grilla ERA5 global regular de "
                    f"(721, 1440), pero se recibió {values.shape} "
                    f"para param={field.metadata('param', default='unknown')}."
                )

            # Convertir longitud de [0, 360) a [-180, 180)
            values = np.roll(
                values,
                -(values.shape[1] // 2),
                axis=1,
            )

            values_n320 = ekr.interpolate(
                values,
                {"grid": (0.25, 0.25)},
                {"grid": "N320"},
            )

            name = _normalise_field_name(
                field=field,
                source=source,
            )

            collected[name].append(values_n320)
    return {
        name: np.stack(time_slices)
        for name, time_slices in collected.items()
    }

In [ ]:
PARAM_SFC = [
    "10u", "10v", "2d", "2t", "msl", "skt",
    "sp", "tcw", "lsm", "z", "slor", "sdor",
]

PARAM_SOIL = [
    "vsw", "sot",
]

PARAM_PL = [
    "z", "t", "u", "v", "w", "q",   # "gh" not used as in era5 z is available for use
]

LEVELS = [
    1000, 925, 850, 700, 600, 500, 400,
    300, 250, 200, 150, 100, 50,
]

SOIL_LEVELS = [1, 2]

In [13]:
def retrieve_raw_initial_conditions(
    *,
    date: dt.datetime,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    """
    Recupera las condiciones iniciales requeridas por AIFS-Single v1.1
    usando ERA5 vía CDS/earthkit.

    La función descarga:
    - variables de superficie,
    - variables en niveles de presión,
    - variables de suelo.

    Retorna
    -------
    tuple[dict[str, np.ndarray], dict[str, np.ndarray]]
        raw_fields:
            Diccionario con variables principales:
            superficie + niveles de presión.

        raw_soil:
            Diccionario con variables de suelo.
    """

    print(f"Descargando campos ERA5/CDS para fecha inicial: {date:%Y-%m-%d %H:%M UTC}")

    # ------------------------------------------------------------
    # 1. Surface fields
    # ------------------------------------------------------------
    print("Descargando variables de superficie...")

    raw_sfc = get_data(
        date=date,
        source="sfc",
        param=PARAM_SFC,
    )

    print(f"Variables de superficie descargadas: {len(raw_sfc)}")

    # ------------------------------------------------------------
    # 2. Pressure-level fields
    # ------------------------------------------------------------
    print("Descargando variables en niveles de presión...")

    raw_pl = get_data(
        date=date,
        source="pl",
        param=PARAM_PL,
        levelist=LEVELS,
    )

    print(f"Variables en niveles de presión descargadas: {len(raw_pl)}")

    # ------------------------------------------------------------
    # 3. Soil fields
    # ------------------------------------------------------------
    print("Descargando variables de suelo...")

    raw_soil = get_data(
        date=date,
        source="soil",
        param=PARAM_SOIL,
        levelist=SOIL_LEVELS,
    )

    print(f"Variables de suelo descargadas: {len(raw_soil)}")

    # ------------------------------------------------------------
    # 4. Merge: surface + pressure levels
    # ------------------------------------------------------------
    raw_fields = {
        **raw_sfc,
        **raw_pl,
    }

    print("Descarga completada.")
    print(f"Campos principales totales: {len(raw_fields)}")
    print(f"Campos de suelo totales: {len(raw_soil)}")

    return raw_fields, raw_soil

In [32]:
raw_fields = None
raw_soil = None

if CONFIG.run_download:
    raw_fields, raw_soil = retrieve_raw_initial_conditions(
        date=DATE,
    )

    print("Campos principales:", len(raw_fields))
    print("Campos de suelo:", len(raw_soil))

    print("\nEjemplo de campos principales:")
    for key in list(raw_fields.keys())[:10]:
        print(key, raw_fields[key].shape)

    print("\nCampos de suelo:")
    for key in raw_soil:
        print(key, raw_soil[key].shape)

else:
    print(
        "Descarga omitida. Activa CONFIG.run_download y ejecuta "
        "de nuevo desde la configuración."
    )

Descargando campos ERA5/CDS para fecha inicial: 2010-06-10 06:00 UTC
Descargando variables de superficie...
Descargando datos para valid date: 2010-06-10 00:00:00
Descargando datos para valid date: 2010-06-10 06:00:00
Variables de superficie descargadas: 12
Descargando variables en niveles de presión...
Descargando datos para valid date: 2010-06-10 00:00:00
Descargando datos para valid date: 2010-06-10 06:00:00
Variables en niveles de presión descargadas: 78
Descargando variables de suelo...
Descargando datos para valid date: 2010-06-10 00:00:00
Descargando datos para valid date: 2010-06-10 06:00:00
Variables de suelo descargadas: 4
Descarga completada.
Campos principales totales: 90
Campos de suelo totales: 4
Campos principales: 90
Campos de suelo: 4

Ejemplo de campos principales:
10u (2, 542080)
10v (2, 542080)
2d (2, 542080)
2t (2, 542080)
msl (2, 542080)
skt (2, 542080)
sp (2, 542080)
tcw (2, 542080)
lsm (2, 542080)
z (2, 542080)

Campos de suelo:
swvl1 (2, 542080)
swvl2 (2, 54208

In [30]:
raw_soil.keys()

dict_keys(['swvl1', 'swvl2', 'stl1', 'stl2'])